In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sys
import os
import seaborn as sns
from scipy import stats as scipystats

# ***import file saved in script5*** 

In [ ]:
################################ import data ############################################################################################

df = pd.read_csv(r"C:\Users\timon\OneDrive\Helmholtz\Scripts\Yagya_jupyter\dfs_for_final_code\ccr4_optimised4MLR_SCD_SCGE_merged_dataset.csv")
df['Birth_or_div_vol_fl'] = df['Birth vol fl'].combine_first(df['Div vol fl'])

df.rename(columns={"abs_growth_rate_flpermin_inclIncompleteCycles": "abs_growth_rate"}, inplace= True)

# ***get relatives' volumes at each frame and calculate volume related parameters*** 

In [3]:
# add volumes of related cells at every frame 
new =[]
import warnings

with warnings.catch_warnings(record=True):
    for date in df['date'].unique():
        date_df  = df[df['date'] == date]
        for pos in date_df['Position'].unique():
            pos_df = date_df[date_df['Position'] == pos]
            for frame in pos_df['frame_i'].unique():
                frame_df = pos_df[pos_df['frame_i'] == frame]
                for cell in (frame_df['Cell_ID']).unique():


                    cell_ind = (frame_df[frame_df['Cell_ID'] == cell]).index
                    if frame_df['cell_cycle_stage'].loc[cell_ind[0]] == "S":

                        rel = frame_df['relative_ID'].loc[cell_ind[0]]

                        rel_ind = (frame_df[frame_df['Cell_ID'] == rel]).index
                        if len(rel_ind) == 0:
                            frame_df.loc[cell_ind[0],'rel_vol_atframe'] = np.nan
                            frame_df.loc[cell_ind[0],'sys_vol_atframe'] = np.nan
                            frame_df.loc[cell_ind[0],'delta_vol_sincePhaseStart_sys'] = np.nan
                            frame_df.loc[cell_ind[0],'delta_vol_sincePhaseStart_mother'] = np.nan

                        else:

                            cell_size_atframe = frame_df['cell_vol_fl'].loc[cell_ind[0]]
                            rel_size_atframe = frame_df['cell_vol_fl'].loc[rel_ind[0]]
                            frame_df.loc[cell_ind[0],'rel_vol_atframe'] = rel_size_atframe
                            frame_df.loc[cell_ind[0],'sys_vol_atframe'] = cell_size_atframe + rel_size_atframe
                            frame_df.loc[cell_ind[0],'delta_vol_sincePhaseStart_mother'] = cell_size_atframe - (frame_df.loc[cell_ind[0],'mother_size_emerg'])
                            frame_df.loc[cell_ind[0],'delta_vol_sincePhaseStart_sys'] = (frame_df.loc[cell_ind[0],'sys_vol_atframe']) - (frame_df.loc[cell_ind[0],'sys_size_emerg'])
                            frame_df.loc[cell_ind[0],'norm_delta_vol_sincePhaseStart_mother'] = (frame_df.loc[cell_ind[0],'delta_vol_sincePhaseStart_mother']) / (frame_df.loc[cell_ind[0],'mother_size_emerg'])
                            frame_df.loc[cell_ind[0],'norm_delta_vol_sincePhaseStart_sys'] = (frame_df.loc[cell_ind[0],'delta_vol_sincePhaseStart_sys']) / (frame_df.loc[cell_ind[0],'sys_size_emerg'])
                            frame_df.loc[cell_ind[0],'bud_vol_atframe/mother_vol_atframe'] = (frame_df.loc[cell_ind[0],'rel_vol_atframe']) / (frame_df.loc[cell_ind[0],'cell_vol_fl'])



                    else:
                        frame_df.loc[cell_ind[0],'rel_vol_atframe'] = np.nan
                        frame_df.loc[cell_ind[0],'sys_vol_atframe'] = np.nan
                        frame_df.loc[cell_ind[0],'delta_vol_sinceG1Start'] = (frame_df.loc[cell_ind[0],'cell_vol_fl']) - (frame_df.loc[cell_ind[0],'Birth_or_div_vol_fl'])
                        frame_df.loc[cell_ind[0],'norm_delta_vol_sinceG1Start'] = (frame_df.loc[cell_ind[0],'delta_vol_sinceG1Start'])/(frame_df.loc[cell_ind[0],'Birth_or_div_vol_fl'])

                    # for stage in frame_df['cell_cycle_stage'].unique():
                    #     stage_df = cycle_df[cycle_df['cell_cycle_stage'] == stage]
                    #
                    #     seconds_in_current_stage =

                new.append(frame_df)



new_df = pd.concat(new).reset_index(drop = True)
new_df = new_df.sort_values(["Unnamed: 0"])
#new_df.drop(columns=['Unnamed: 0.1.1.1','Unnamed: 0', 'Unnamed: 0.1.1.1.1'], axis = 1, inplace = True)
display(new_df.head(5))


,Unnamed: 0,generation_num,Position,Cell_ID,growthmedium,frame_i,cell_cycle_stage,relative_ID,relationship,cell_vol_fl,...,Birth_or_div_vol_fl,rel_vol_atframe,sys_vol_atframe,delta_vol_sincePhaseStart_mother,delta_vol_sincePhaseStart_sys,norm_delta_vol_sincePhaseStart_mother,norm_delta_vol_sincePhaseStart_sys,bud_vol_atframe/mother_vol_atframe,delta_vol_sinceG1Start,norm_delta_vol_sinceG1Start
0,0,0.0,Position_1,1,SCD,0,S,3.0,bud,29.790028,...,NaN,84.336230,114.126258,NaN,NaN,NaN,NaN,2.831022,NaN,NaN
3,1,0.0,Position_1,1,SCD,1,S,3.0,bud,33.309480,...,NaN,85.824604,119.134083,NaN,NaN,NaN,NaN,2.576582,NaN,NaN
6,2,0.0,Position_1,1,SCD,2,S,3.0,bud,36.152073,...,NaN,84.944725,121.096798,NaN,NaN,NaN,NaN,2.349650,NaN,NaN
9,3,0.0,Position_1,1,SCD,3,S,3.0,bud,39.181156,...,NaN,86.185197,125.366353,NaN,NaN,NaN,NaN,2.199659,NaN,NaN
12,4,0.0,Position_1,1,SCD,4,S,3.0,bud,41.634529,...,NaN,88.021124,129.655653,NaN,NaN,NaN,NaN,2.114138,NaN,NaN


# ***absolute growth rate calculation for cycles with complete G1 but incomplete S*** 

In [4]:
# absolute growth rate calculation for cycles with complete G1 but incomplete S 
new1 = []
for date in new_df['date'].unique():
    date_df  = new_df[new_df['date'] == date]
    for pos in date_df['Position'].unique():
        pos_df = date_df[date_df['Position'] == pos]
        for cell in pos_df['Cell_ID'].unique():
            cell_df = pos_df[pos_df['Cell_ID'] == cell]
            for cycle in cell_df['generation_num'].unique():
                cycle_df =  cell_df[cell_df['generation_num'] == cycle]


                frame_min = cycle_df['frame_i'].min()
                min_frame_index = cycle_df[cycle_df['frame_i'] == frame_min].index
                vol_min = cycle_df['cell_vol_fl'].loc[min_frame_index[0],]

                frame_max = cycle_df['frame_i'].max()
                max_frame_index = cycle_df[cycle_df['frame_i'] == frame_max].index

                if ((cycle_df['cell_cycle_stage'].loc[max_frame_index[0],] == "S") &
                    (np.isnan(cycle_df['sys_size_div'].loc[max_frame_index[0],]) == True) &
                    (cycle_df['generation_num'].loc[max_frame_index[0],] != 0)):
                    vol_max = cycle_df['sys_vol_atframe'].loc[max_frame_index[0],]

                    delta_vol_fl = vol_max-vol_min
                    delta_time_mins = (frame_max - frame_min)*3
                    abs_growth_rate_flpermin = delta_vol_fl/delta_time_mins

                    cycle_df['delta_vol_fl'] = delta_vol_fl
                    cycle_df['delta_time_mins'] = delta_time_mins
                    cycle_df['abs_growth_rate'] = abs_growth_rate_flpermin





                new1.append(cycle_df)

new1_df = pd.concat(new1).reset_index(drop = True)

new1_df.drop(columns=['Unnamed: 0'], axis = 1, inplace = True)

#display(new1_df.head(50))

C:\Users\yagya.chadha\AppData\Local\Temp\ipykernel_14884\88203161.py:29: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  cycle_df['delta_vol_fl'] = delta_vol_fl
C:\Users\yagya.chadha\AppData\Local\Temp\ipykernel_14884\88203161.py:30: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  cycle_df['delta_time_mins'] = delta_time_mins
C:\Users\yagya.chadha\AppData\Local\Temp\ipykernel_14884\88203161.py:31: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_inde

# ***assign daughter or not and calculate time in stage (mins)*** 
# ***save df*** 

In [5]:
############################################################ binary daughter or not #################################################################################
new1_df['Daughterhood_binary'] = 0
for cycle in new1_df['generation_num'].unique():
    ind = (new1_df[new1_df['generation_num'] == cycle]).index
    if cycle == 1:
        new1_df['Daughterhood_binary'].loc[ind] = 1
#display(new1_df.head(100))


new2 = []
import warnings

with warnings.catch_warnings(record=True):
    
    for date in new1_df['date'].unique():
        date_df  = new1_df[new1_df['date'] == date]
        for pos in date_df['Position'].unique():
            pos_df = date_df[date_df['Position'] == pos]
            for cell in pos_df['Cell_ID'].unique():
                cell_df = pos_df[pos_df['Cell_ID'] == cell]
                for cycle in cell_df['generation_num'].unique():
                    cycle_df =  cell_df[cell_df['generation_num'] == cycle]
                    for stage in cycle_df['cell_cycle_stage'].unique():
                        stage_df = cycle_df[cycle_df['cell_cycle_stage'] == stage]
                        frame_at_stage_entry = stage_df['frame_i'].min()
                        for ind in stage_df.index:
                            stage_df.loc[ind,'time_in_stage_mins'] = ((stage_df['frame_i'].loc[ind]) - (frame_at_stage_entry))*3

                        new2.append(stage_df)



new2_df = pd.concat(new2).reset_index(drop = True)
#display(new2_df.head(100))

# new2_df.to_csv(r"C:\Users\yagya.chadha\Desktop\Yagya\Analysed Live cell microscopy data\optimised4MLR_SCD_SCGE_merged_dataset")


C:\Users\yagya.chadha\AppData\Local\Temp\ipykernel_14884\4070377345.py:6: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  new1_df['Daughterhood_binary'].loc[ind] = 1
C:\Users\yagya.chadha\AppData\Local\Temp\ipykernel_14884\4070377345.py:6: Set

In [ ]:
optimised4MLR_SCDandSCGE_dataset = pd.read_csv(r"C:\Users\timon\OneDrive\Helmholtz\Scripts\Yagya_jupyter\dfs_for_final_code\optimised4MLR_SCD_SCGE_merged_dataset.csv").reset_index()

complete_optimised4MLR_SCDandSCGE_dataset = pd.concat([optimised4MLR_SCDandSCGE_dataset, new2_df])
complete_optimised4MLR_SCDandSCGE_dataset.to_csv(r"C:\Users\timon\OneDrive\Helmholtz\Scripts\Yagya_jupyter\dfs_for_final_code\complete_optimised4MLR_SCD_SCGE_merged_dataset.csv")